# PageGauge A100 cross-GPU confirmation
Select **Runtime > Change runtime type > A100 GPU** before running. Colab does not guarantee GPU type; the setup fails closed unless it sees a full A100 compute-capability 8.0 device with at least 38 GiB memory and 100 SMs.


In [ ]:
!nvidia-smi

Upload `pagegauge_a100_colab.zip`. The archive contains code and the frozen WikiText artifact, but not model weights.


In [ ]:
from pathlib import Path

from google.colab import files

uploaded = files.upload()
archives = [name for name in uploaded if name.endswith(".zip")]
assert len(archives) == 1, archives
archive = archives[0]
!rm -rf /content/pagegauge_a100_run
!mkdir -p /content/pagegauge_a100_run
!unzip -q "{archive}" -d /content/pagegauge_a100_run
%cd /content/pagegauge_a100_run/pagegauge_a100_colab

Install pinned dependencies, validate the A100, download the pinned Mistral snapshot, and prepare the audited FlashInfer 0.6.17 include tree. Set `HF_TOKEN` first only if Hugging Face requests authentication.


In [ ]:
!bash a100_colab/setup_colab.sh

Compile for `sm_80` in a clean cache and run the long-context factorization-vs-explicit-reconstruction gate.


In [ ]:
!bash a100_colab/run_kernel_smoke.sh

Run the eight fresh-process ABBA/BAAB blocks. This is the long cell; keep the browser connected and do not run another GPU cell concurrently.


In [ ]:
!bash a100_colab/run_cross_gpu_confirmation.sh

In [ ]:
import json

result_path = Path("results/a100_cross_gpu_s4_a128_t768/final_a100_analysis.json")
result = json.loads(result_path.read_text())
primary = result["aggregates"]["cache_neutral"]["wall_ms"]
print(
    json.dumps(
        {
            "passed": result["passed"],
            "speedup": primary["speedup_geomean"],
            "hierarchical_95_ci": primary["hierarchical_fixture_pair_bootstrap_95_ci"],
            "served_cache": result["served_cache"],
        },
        indent=2,
    )
)
!zip -qr /content/pagegauge_a100_evidence.zip results/a100_cross_gpu_s4_a128_t768
files.download("/content/pagegauge_a100_evidence.zip")